In [1]:
import pdal
from pathlib import Path
import json

### LiDAR Preprocessing and DEM Generation

File search

In [2]:
# Current working directory absolute path
data_dir = Path().resolve()
# Recursive search of *.laz or *.las files in all sub-folders
las_files = list(data_dir.rglob("*.laz")) + list(data_dir.rglob("*.las"))
las_files = [str(f) for f in las_files]
print(f"Found {len(las_files)} files")

Found 10 files


Define DEM parameters

In [3]:
# DEM parameters
resolution = 0.5
output_type = "min" # or "idw", but "min" probably better for our purpose
radius = 1.0 # 1 m, we can try different values
power = 2.0 # only for idw
window_size = 5 # if there are no points in radius, how many surrounding values use for interpolation

Set output directiories

In [5]:
dem_dir = data_dir / "inference_dem_tiles"

# Create data folders if they don't exist
dem_dir.mkdir(parents=True, exist_ok=True)

Process each file using PDAL pipeline and save as DEM tiles

In [6]:
# For each file (enumerate just so we can track how many files has been processed)
for i, las in enumerate(las_files, 1):
    las_path = Path(las)
    # name of DEM file
    dem_file = dem_dir / f"{las_path.stem}_dem_{output_type}.tif"
    # Tracking progress
    print(f"[{i}/{len(las_files)}] Processing {las_path.name} → {dem_file.name}")

    # input for PDAL is a JSON which can be done as a dictionary in python and then converting to JSON
    pipeline_dict = {
        "pipeline": [{"type": "readers.las", "filename": str(las_path)}, # read file     
            {"type": "filters.range", "limits": "Classification[2:2]"},  # only ground class
            {"type": "filters.outlier", "method": "statistical", "mean_k": 8, "multiplier": 2.5}, # filter outlying points
            {
                "type": "writers.gdal",  # create DEM with chosen parameters
                "filename": str(dem_file),
                "resolution": resolution,
                "output_type": output_type,
                "radius": radius,
                "power": power,
                "window_size": window_size,
                "gdaldriver": "GTiff"
            }
        ]
    }
    # convert dictionary to JSON and run pipeline
    pipeline = pdal.Pipeline(json.dumps(pipeline_dict))
    count = pipeline.execute()

print(f"Pipeline finished. Created {len(las_files)} DEM files")

[1/10] Processing P4433B2_3.laz → P4433B2_3_dem_min.tif
[2/10] Processing P4433B2_4.laz → P4433B2_4_dem_min.tif
[3/10] Processing P4433B2_6.laz → P4433B2_6_dem_min.tif
[4/10] Processing P4433B2_7.laz → P4433B2_7_dem_min.tif
[5/10] Processing P4433B3_1.laz → P4433B3_1_dem_min.tif
[6/10] Processing P4433B3_2.laz → P4433B3_2_dem_min.tif
[7/10] Processing P4433B3_4.laz → P4433B3_4_dem_min.tif
[8/10] Processing P4433B3_7.laz → P4433B3_7_dem_min.tif
[9/10] Processing P4433B3_9.laz → P4433B3_9_dem_min.tif
[10/10] Processing P4433B4_9.laz → P4433B4_9_dem_min.tif
Pipeline finished. Created 10 DEM files
